# İP2 — Veri Temizleme ve Master Tablo Oluşturma
**Sorumlu:** Sudenaz Güven | **Tarih:** 19 Haziran 2026

Bu notebook 7 fakülteye ait ham xlsx dosyalarını temizler, birleştirir ve `master_clean.xlsx` çıktısını üretir.

### Kullanılan Kütüphaneler ve Neden Kullanıldıkları

| Kütüphane | Kullanım Amacı |
|-----------|----------------|
| `pandas`  | Veri okuma, birleştirme, temizleme ve kaydetme |
| `numpy`   | Sayısal işlemler ve NaN yönetimi |
| `openpyxl`| `.xlsx` dosyalarını okumak/yazmak için pandas backend |

In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Yollar ──────────────────────────────────────────────────────────────
RAW = Path("../data/raw")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

# ── Ham dosya → fakülte adı eşleşmesi ───────────────────────────────────
FILES = {
    "Mühendislik":      "Mu_hendislik.xlsx",
    "Güzel Sanatlar":   "Gu_zel_Sanatlar.xlsx",
    "Fen-Edebiyat":     "FenEdebiyat.xlsx",
    "Eğitim":           "Eg_itim.xlsx",
    "Siyasal Bilgiler": "Siyasal_Bilgiler.xlsx",
    "Spor Bilimleri":   "Spor_Bilimleri.xlsx",
    "Teknoloji":        "Teknoloji.xlsx",
}

# ── Fakülte → alan grubu eşleşmesi ──────────────────────────────────────
ALAN_GRUBU = {
    "Mühendislik":      "STEM",
    "Teknoloji":        "STEM",
    "Fen-Edebiyat":     "Fen-Sosyal",
    "Eğitim":           "Sosyal",
    "Siyasal Bilgiler": "Sosyal",
    "Güzel Sanatlar":   "Sanat",
    "Spor Bilimleri":   "Spor",
}

# ── Etkinlik sütun adları ────────────────────────────────────────────────
ETK_COLS = [f"EtkinlikAd_{i}" for i in range(1, 22)]
KAZ_COLS = [f"OgrenmeKazanim_{i}" for i in range(1, 26)]

print("Sabitler ve yollar tanımlandı.")
print(f"  RAW klasörü : {RAW}")
print(f"  OUT klasörü : {OUT}")

Sabitler ve yollar tanımlandı.
  RAW klasörü : ../data/raw
  OUT klasörü : ../data/processed


### Adım 1 — Ham Dosyaları Yükleme

7 farklı fakülteye ait `.xlsx` dosyaları `pandas.read_excel()` ile okunur.  
Her dosyaya hangi fakülteden geldiğini gösteren `Fakulte` sütunu eklenir.  
`pd.concat()` ile tüm fakülteler tek bir DataFrame'de birleştirilir.  

> **Beklenen sonuç:** ~5.807 satır

In [2]:
dfs = []
for fakulte, fname in FILES.items():
    path = RAW / fname
    df_tmp = pd.read_excel(path)
    df_tmp["Fakulte"] = fakulte
    dfs.append(df_tmp)
    print(f"  Yüklendi: {fname:<30}  →  {len(df_tmp):>5} satır")

df = pd.concat(dfs, ignore_index=True)
raw_count = len(df)

print(f"\nToplam birleşik satır : {raw_count}")
print(f"Toplam sütun         : {len(df.columns)}")

  Yüklendi: Mu_hendislik.xlsx               →   1623 satır


  Yüklendi: Gu_zel_Sanatlar.xlsx            →   1149 satır


  Yüklendi: FenEdebiyat.xlsx                →    978 satır
  Yüklendi: Eg_itim.xlsx                    →    641 satır


  Yüklendi: Siyasal_Bilgiler.xlsx           →    568 satır
  Yüklendi: Spor_Bilimleri.xlsx             →    442 satır
  Yüklendi: Teknoloji.xlsx                  →    406 satır



Toplam birleşik satır : 5807
Toplam sütun         : 76


### Adım 2 — Veri Temizleme

Üç temel temizleme adımı uygulanır:

1. **AKTS = 0 olan satırlar silinir** — Bu değer veri girişi hatasına işaret eder; kredisi 0 olan ders analitik olarak anlamsızdır.
2. **`OgrenmeKazanim_1` boş olan satırlar silinir** — En az bir öğrenme kazanımı olmayan dersler Bloom analizi yapılamayacağından çıkarılır.
3. **`EtkinlikAd` sütunlarında `str.strip()` uygulanır** — Baştaki/sondaki boşluklar etkinlik türü eşleştirmesini bozabileceğinden temizlenir.

In [3]:
# 1. AKTSKredi = 0 olan satırları sil
onceki = len(df)
df = df[df["AKTSKredi"] != 0]
drop_akts = onceki - len(df)
print(f"AKTSKredi=0 düşülen satır : {drop_akts}")

# 2. OgrenmeKazanim_1 boş olan satırları sil
onceki = len(df)
df = df[df["OgrenmeKazanim_1"].notna()]
df = df[df["OgrenmeKazanim_1"].astype(str).str.strip().str.lower() != "nan"]
drop_kaz = onceki - len(df)
print(f"Kazanım boş düşülen satır  : {drop_kaz}")

# 3. EtkinlikAd sütunlarında str.strip()
for c in ETK_COLS:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().replace("nan", pd.NA)
print(f"EtkinlikAd sütunları temizlendi.")

print(f"\nTemizlik sonrası satır     : {len(df)}")

AKTSKredi=0 düşülen satır : 57
Kazanım boş düşülen satır  : 12
EtkinlikAd sütunları temizlendi.

Temizlik sonrası satır     : 5738


### Adım 3 — İkinci Öğretim (İÖ) Tekilleştirme

EBS sisteminde aynı ders hem **Normal** hem de **İkinci Öğretim (İÖ)** programına ayrı satırlar hâlinde kayıtlı gelebilir.  
İçerik özdeş olduğundan bu tekrarlar analizde çift sayıma yol açar.

**Strateji:**
- Aynı `Katalogid` için Normal öğretim satırı varsa → İÖ satırı **silinir**
- Yalnızca İÖ satırı olan dersler (özgün İÖ dersleri) → **korunur**

Bu adımda yaklaşık **~1.600 satır** düşürülür.

In [4]:
# BolumAd içinde "(İÖ)" veya "(IO)" geçen satırları işaretle
df["is_io"] = df["BolumAd"].astype(str).str.contains(
    r"\(İÖ\)|\(IO\)", regex=True, na=False
)

before_dedup = len(df)

# is_io=False önce gelecek şekilde sırala → keep='first' ile Normal öğretim korunur
df = df.sort_values("is_io").drop_duplicates(subset="Katalogid", keep="first")

drop_io = before_dedup - len(df)
print(f"İÖ tekrar düşülen satır  : {drop_io}")
print(f"Tekilleştirme sonrası    : {len(df)} satır")
print(f"  Normal öğretim         : {(~df['is_io']).sum()}")
print(f"  Özgün İÖ dersleri      : {df['is_io'].sum()}")

İÖ tekrar düşülen satır  : 1605
Tekilleştirme sonrası    : 4133 satır
  Normal öğretim         : 4133
  Özgün İÖ dersleri      : 0


### Adım 4 — Öznitelik Mühendisliği

Ham veriden analitik değeri yüksek yeni özellikler türetilir:

| Özellik | Açıklama |
|---------|----------|
| `n_ogrenme_kazanim` | Her ders için dolu kazanım sayısı (1–25) |
| `kazanim_concat` | Tüm kazanımlar `" \| "` ile birleştirilmiş metin — İP3 NLP analizi için |
| `n_etkinlik` | Dolu etkinlik türü sayısı |
| `has_lab` | Laboratuvar etkinliği var mı? (0/1) |
| `has_proje` | Proje etkinliği var mı? (0/1) |
| `has_uygulama` | Uygulama etkinliği var mı? (0/1) |
| `has_atolye` | Atölye etkinliği var mı? (0/1) |
| `has_odev` | Ödev etkinliği var mı? (0/1) |
| `has_sinav` | Sınav/Quiz etkinliği var mı? (0/1) |
| `has_sunum` | Sunum/Seminer etkinliği var mı? (0/1) |
| `alan_grubu` | Fakülteye göre alan: STEM, Sosyal, Fen-Sosyal, Sanat, Spor |

In [5]:
# Alan grubu
df["alan_grubu"] = df["Fakulte"].map(ALAN_GRUBU)

# Kazanım sayısı ve birleşik metin
def count_kaz(row):
    return sum(1 for c in KAZ_COLS
               if c in row and pd.notna(row[c])
               and str(row[c]).strip().lower() not in ("", "nan"))

def concat_kaz(row):
    vals = [str(row[c]).strip() for c in KAZ_COLS
            if c in row and pd.notna(row[c])
            and str(row[c]).strip().lower() not in ("", "nan")]
    return " | ".join(vals)

df["n_ogrenme_kazanim"] = df.apply(count_kaz, axis=1)
df["kazanim_concat"]    = df.apply(concat_kaz, axis=1)

# Etkinlik bayrakları
def etkinlik_features(row):
    vals = " | ".join([
        str(row[c]).strip().lower()
        for c in ETK_COLS
        if c in row and pd.notna(row[c])
        and str(row[c]).strip().lower() not in ("", "nan")
    ])
    n = vals.count("|") + 1 if vals.strip() else 0
    return pd.Series({
        "n_etkinlik":   n,
        "has_lab":      int("laboratuvar" in vals),
        "has_proje":    int("proje" in vals),
        "has_uygulama": int("uygulama" in vals),
        "has_atolye":   int("atölye" in vals or "atolye" in vals),
        "has_odev":     int("ödev" in vals or "odev" in vals),
        "has_sinav":    int("sınav" in vals or "sinav" in vals or "quiz" in vals),
        "has_sunum":    int("sunum" in vals or "seminer" in vals),
    })

ef = df.apply(etkinlik_features, axis=1)
df = pd.concat([df, ef], axis=1)
df.reset_index(drop=True, inplace=True)

print(f"Öznitelikler eklendi.")
print(f"Toplam sütun sayısı      : {len(df.columns)}")
print(f"Ort. kazanım / ders      : {df['n_ogrenme_kazanim'].mean():.2f}")
print(f"Ort. etkinlik / ders     : {df['n_etkinlik'].mean():.2f}")

Öznitelikler eklendi.
Toplam sütun sayısı      : 88
Ort. kazanım / ders      : 5.29
Ort. etkinlik / ders     : 4.38


### Sonuç

| | Değer |
|---|---|
| **Girdi** | 5.807 satır (7 fakülte, ham) |
| **Çıktı** | 4.133 satır, 65+ sütun |
| **Dosya** | `data/processed/master_clean.xlsx` ve `master_clean.csv` |

Oluşturulan master tablo İP3 Bloom NLP analizinin girdisini oluşturur.

In [6]:
df.to_excel(OUT / "master_clean.xlsx", index=False)
df.to_csv(OUT / "master_clean.csv", index=False, encoding="utf-8-sig")

n_rows = len(df)
n_cols = len(df.columns)

print("=" * 48)
print("            TEMİZLİK RAPORU")
print("=" * 48)
print(f"  Ham satır              : {raw_count}")
print(f"  AKTSKredi=0 düşülen   : {drop_akts}")
print(f"  Kazanım boş düşülen   : {drop_kaz}")
print(f"  İÖ tekrar düşülen     : {drop_io}")
print(f"  MASTER SATIR           : {n_rows}")
print(f"  MASTER KOLON           : {n_cols}")

print("\n── Fakülte Dağılımı ─────────────────────────")
for fak, cnt in df["Fakulte"].value_counts().items():
    print(f"  {fak:<22}: {cnt:>5}  ({cnt/n_rows*100:.1f}%)")

print("\n── Etkinlik Oranları ────────────────────────")
for col in ["has_lab","has_proje","has_uygulama","has_atolye","has_odev","has_sinav","has_sunum"]:
    s = df[col].sum()
    print(f"  {col:<18}: {s:>5}  ({s/n_rows*100:.1f}%)")

print("\n── Diğer ────────────────────────────────────")
print(f"  Ort. n_etkinlik        : {df['n_etkinlik'].mean():.2f}")
print(f"  Ort. n_kazanim         : {df['n_ogrenme_kazanim'].mean():.2f}")
print(f"  Ort. AKTSKredi         : {df['AKTSKredi'].mean():.2f}")
print(f"\n  Kaydedildi → {OUT / 'master_clean.xlsx'}")
print(f"  Kaydedildi → {OUT / 'master_clean.csv'}")

            TEMİZLİK RAPORU
  Ham satır              : 5807
  AKTSKredi=0 düşülen   : 57
  Kazanım boş düşülen   : 12
  İÖ tekrar düşülen     : 1605
  MASTER SATIR           : 4133
  MASTER KOLON           : 88

── Fakülte Dağılımı ─────────────────────────
  Mühendislik           :  1090  (26.4%)
  Fen-Edebiyat          :   890  (21.5%)
  Güzel Sanatlar        :   710  (17.2%)
  Spor Bilimleri        :   431  (10.4%)
  Eğitim                :   366  (8.9%)
  Teknoloji             :   323  (7.8%)
  Siyasal Bilgiler      :   323  (7.8%)

── Etkinlik Oranları ────────────────────────
  has_lab           :   139  (3.4%)
  has_proje         :   303  (7.3%)
  has_uygulama      :   382  (9.2%)
  has_atolye        :    20  (0.5%)
  has_odev          :   723  (17.5%)
  has_sinav         :  4039  (97.7%)
  has_sunum         :   312  (7.5%)

── Diğer ────────────────────────────────────
  Ort. n_etkinlik        : 4.38
  Ort. n_kazanim         : 5.29
  Ort. AKTSKredi         : 4.27

  Kaydedildi 